# CV04 · Colab GPU training · compact disk saves and recovery

**Training runs on the remote Colab GPU through VS Code.** Checkpoints, exports, logs and reports are written to **Colab's `/content` disk**. Your Mac is only the notebook editor and the destination for downloaded backup ZIPs. No new training outputs are written to Drive.

This version uses your already-uploaded v7 training code and shared dataset. It copies your existing Drive run **without deleting or changing it**, so folds 1–4 are retained and fold 5 can resume from its saved checkpoint.

**Saving policy:** one rolling resumable focal checkpoint per active fold, saved every epoch on Colab disk; completed folds retain exact full-precision weights, their small contrastive/KNN checkpoint, configuration and reports. A compressed ZIP is prepared for each completed fold. ZIP compression is lossless; dense weights may only compress modestly. Quantization is not silently applied.

**Backup limit:** Colab disk is temporary and can disappear when the VM is deleted. A ZIP under `/content` is still temporary until you download it. The default **Run All trains all remaining folds**, preparing a ZIP after each fold, then performs selection and final export. Download completed ZIPs while training continues. Set `FOLDS_PER_RUN = 1` if you prefer to pause after each fold for a download. Results since your last download are not durably backed up. [Colab runtime lifetime](https://research.google.com/colaboratory/faq.html).

1. Open this notebook in VS Code and keep your **Colab GPU kernel** selected.
2. Run setup and import your existing saved run.
3. Before continuing training, download the ZIPs for the four previously completed folds (prepared below).
4. Run the training cell to finish all remaining folds, and download completed fold ZIPs from the server while it runs. Selection waits until every requested fold is complete.

To download in VS Code, enable Colab's **Server Mounting** setting if needed, run **Colab: Mount Server To Workspace…**, refresh its Explorer, then right-click an export ZIP and select **Download**. This copies files to your Mac; the training computation stays on Colab. [Official download guidance](https://github.com/googlecolab/colab-vscode/issues/267).

In [ ]:
# 1a. Connect to the remote Colab GPU and mount Drive.
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile

try:
    from google.colab import drive
except ImportError:
    raise RuntimeError('In VS Code select Kernel → Colab → New Colab Server → GPU.')

drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/ParkinsonResearch')
PROJECT_DIR = Path('/content/ParkinsonCV04Compact')
DATA_ROOT = Path('/content/ParkinsonData')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU Colab server in VS Code, then rerun setup.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 1b. Verify and load the small training-code ZIP from Drive.
CODE_NAME = 'parkinson_training_code_v7_cv04_storage_fix.zip'
CODE_SHA256 = '4d4104e19c7e8f9671255dd19aa90ab97cbd6623aa28352775adcb149ef33dc6'

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def extract_checked(source, destination, expected_sha256):
    if not source.is_file():
        raise FileNotFoundError(f'Copy the supplied ZIP to {source}, then rerun this cell.')
    local_zip = Path('/content') / source.name
    if source.resolve() != local_zip.resolve():
        shutil.copy2(source, local_zip)
    if sha256(local_zip) != expected_sha256:
        raise ValueError(f'Checksum differs from this notebook: {source.name}')
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(local_zip) as archive:
        for name in archive.namelist():
            if not (destination / name).resolve().is_relative_to(destination.resolve()):
                raise ValueError(f'Unsafe archive path: {name}')
        archive.extractall(destination)

# Check common upload locations on the remote server and mounted Drive.
archive_locations = [
    DRIVE_DIR / CODE_NAME,
    Path('/content') / CODE_NAME,
    DRIVE_DIR.parent / CODE_NAME,
    DRIVE_DIR.parent / 'Downloads' / CODE_NAME,
    DRIVE_DIR / 'Downloads' / CODE_NAME,
]
code_zip = next((path for path in archive_locations if path.is_file()), None)
if code_zip is None:
    checked = '\n'.join(str(path) for path in archive_locations)
    raise FileNotFoundError(
        f'The code ZIP is not visible to this Colab kernel yet.\n'
        f'Upload {CODE_NAME} to My Drive/ParkinsonResearch, then rerun this cell.\n'
        f'Checked:\n{checked}'
    )
print('Loading code ZIP:', code_zip)
extract_checked(code_zip, PROJECT_DIR, CODE_SHA256)
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('Readable Python source files:', PROJECT_DIR)

# Refresh progress helpers when rerunning setup in an existing kernel.
import importlib
for module_name in ['utils.notebook_training', 'utils.cv04_notebook_progress']:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])


In [ ]:
# 1c. Load and verify the existing shared dataset; keep Colab's GPU libraries.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'Pillow>=10,<13', 'numpy>=1.26,<3', 'openpyxl>=3.1,<4', 'tqdm>=4.66,<5'], check=True)
from tqdm.auto import tqdm

extract_checked(
    DRIVE_DIR / 'datasets/parkinson_dataset_v1.zip', DATA_ROOT,
    '85efa52f80adbd357537842146ae107e3cd6b9817eb7ec95beba2644d813f1c6',
)
metadata = json.loads((DATA_ROOT / 'dataset_package.json').read_text())
for name, info in tqdm(metadata['files'].items(), desc='Verifying dataset', unit='file'):
    target = (DATA_ROOT / name).resolve()
    if not target.is_relative_to(DATA_ROOT.resolve()) or sha256(target) != info['sha256']:
        raise ValueError(f'Dataset verification failed: {name}')
MANIFEST = DATA_ROOT / 'datasets/annotations/fold_manifest.csv'
from utils.cv04_notebook_progress import run_cv04_with_progress
print('Ready. Configure the fold range below, then run training.')

### Local disk helpers
These readable functions copy and verify the existing run, adjust checkpoint file locations after a copy, and create lossless ZIPs. The original Drive run stays intact. Restored archives must be extracted into the same run directory, then have their source-backbone paths refreshed before independent reload.

In [ ]:
# Disk helpers: verified import and portable checkpoint paths.
from tqdm.auto import tqdm

def rebase_backbone_paths(run_dir):
    from utils.representation_pipeline import atomic_checkpoint
    changed = 0
    for checkpoint in run_dir.rglob('*.pt'):
        if checkpoint.name not in {'last.pt', 'selected.pt'}:
            continue
        saved = torch.load(checkpoint, map_location='cpu', weights_only=True)
        source = saved.get('source_baseline')
        if not source:
            continue
        baseline = run_dir / 'baselines' / f"fold{source['fold_id']:02d}.pt"
        if not baseline.is_file() or sha256(baseline) != source['sha256']:
            raise ValueError(f'Missing or changed source backbone: {baseline}')
        if source['path'] != str(baseline.resolve()):
            source['path'] = str(baseline.resolve())
            atomic_checkpoint(checkpoint, saved)
            changed += 1
    return changed

def import_saved_run(source, destination):
    if source.resolve() == destination.resolve():
        raise ValueError('Import source and destination must differ.')
    if (destination / 'recipe.json').is_file():
        print('Keeping the current Colab-disk run; not overwriting it from Drive.')
        return
    if not (source / 'recipe.json').is_file():
        print('No prior Drive run found; the training cell can start a fresh local run.')
        return
    sources = [p for p in source.rglob('*') if p.is_file() and not p.name.endswith('.tmp')]
    needed = sum(p.stat().st_size for p in sources) + 1_000_000_000
    if shutil.disk_usage(destination.parent).free < needed:
        raise RuntimeError('Colab disk needs more space to import and verify the existing run.')
    staging = destination.with_name(destination.name + '.importing')
    staging.mkdir(parents=True, exist_ok=True)
    for original in tqdm(sources, desc='Copying saved run to Colab disk', unit='file'):
        target = staging / original.relative_to(source)
        expected = sha256(original)
        if target.is_file() and sha256(target) == expected:
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        temporary = target.with_name(target.name + '.copying')
        shutil.copy2(original, temporary)
        if sha256(temporary) != expected:
            raise RuntimeError(f'Import verification failed: {original}')
        os.replace(temporary, target)
    if destination.exists():
        raise RuntimeError(f'Import destination is nonempty: {destination}')
    # Source checkpoint bytes were verified before making the copied run portable.
    os.replace(staging, destination)
    rebase_backbone_paths(destination)
    print('Existing run imported and verified. Drive files are unchanged.')

def brief_training_error(text):
    import re
    lines = text.splitlines()
    roots = [i for i, line in enumerate(lines)
             if (re.match(r'^[\w.]*(?:Error|Exception):', line)
                 and 'CalledProcessError:' not in line and 'TrainingProcessError:' not in line)
             or line.startswith(('Resume ', 'Only a ', 'This checkpoint'))]
    index = roots[-1] if roots else max(0, len(lines) - 12)
    return '\n'.join(lines[max(0, index - 3):index + 3])



def prepare_active_checkpoint(run_dir, drive_run, fold, manifest, quarantine_dir):
    # Never replace a completed backbone or mutate a Drive file.
    if (run_dir / 'baselines' / f'fold{fold:02d}.pt').is_file():
        return 'completed_backbone'
    work = run_dir / 'work' / f'fold{fold:02d}'
    candidates = list((work / 'checkpoints').glob('*_last.pt'))
    if not candidates:
        return 'fresh_fold'
    if len(candidates) != 1:
        raise ValueError(f'Ambiguous resume files in {work}; existing files preserved.')
    current = candidates[0]
    manifest_hash = sha256(manifest)

    def checked_epoch(path):
        saved = torch.load(path, map_location='cpu', weights_only=True)
        if (saved.get('checkpoint_kind') != 'last' or saved.get('fold_id') != fold
                or saved.get('fold_manifest_sha256') != manifest_hash
                or 'model_state_dict' not in saved or 'optimizer_state_dict' not in saved):
            raise ValueError(f'Resume metadata does not match fold {fold}; file preserved: {path}')
        return int(saved['epoch'])

    def unreadable(error):
        return isinstance(error, (EOFError, zipfile.BadZipFile)) or any(
            word in str(error) for word in ['PytorchStreamReader', 'pickle data was truncated',
                                          'invalid load key', 'failed finding central directory'])

    try:
        epoch = checked_epoch(current)
        print(f'Fold {fold}: checkpoint verified, resume after epoch {epoch}.')
        return 'valid_checkpoint'
    except Exception as error:
        if not unreadable(error):
            raise
        print(f'Fold {fold}: unreadable local checkpoint: {str(error)[:160]}')

    # Try the unchanged Drive source before giving up the active fold's progress.
    original = drive_run / current.relative_to(run_dir)
    fallback_epoch = None
    if original.is_file():
        try:
            fallback_epoch = checked_epoch(original)
        except Exception as error:
            if not unreadable(error):
                raise
            print('The Drive source checkpoint is also unreadable.')
    quarantine_dir.mkdir(parents=True, exist_ok=True)
    attempt = 1
    backup = quarantine_dir / f'fold{fold:02d}_attempt{attempt}'
    while backup.exists():
        attempt += 1
        backup = quarantine_dir / f'fold{fold:02d}_attempt{attempt}'
    if fallback_epoch is not None:
        temporary = current.with_suffix('.pt.recovery')
        shutil.copy2(original, temporary)
        if sha256(temporary) != sha256(original):
            raise RuntimeError('Recovery copy verification failed; original local file preserved.')
        checked_epoch(temporary)
        backup.mkdir()
        os.replace(current, backup / current.name)
        os.replace(temporary, current)
        print(f'Restored a valid Drive copy at epoch {fallback_epoch}. Bad local copy retained at {backup}.')
        return 'restored_checkpoint'
    # No usable checkpoint remains. Preserve the whole local unfinished work folder
    # outside the run, then let the trainer start only this unfinished fold again.
    os.replace(work, backup)
    print(f'No readable checkpoint for fold {fold}; restarting this unfinished fold from epoch 1.')
    print(f'Unreadable local files retained at {backup}. Completed folds and Drive files are unchanged.')
    return 'restart_unfinished_fold'


In [ ]:
# Disk helpers: completed-fold detection and lossless backups.
def completed_folds(run_dir, first, last, projection_epochs):
    finished = []
    for fold in range(first, last + 1):
        stage = run_dir / 'projections/fixed_recipe' / f'fold{fold:02d}' / 'last.pt'
        if not stage.is_file() or not (run_dir / 'baselines' / f'fold{fold:02d}.pt').is_file():
            continue
        saved = torch.load(stage, map_location='cpu', weights_only=True)
        if saved['epoch'] == projection_epochs == saved['epochs_requested']:
            if saved['source_baseline']['fold_id'] != fold:
                raise ValueError('Projection fold metadata mismatch.')
            finished.append(fold)
    return finished

def export_backup(run_dir, export_dir, *, folds=(), reports_only=False):
    export_dir.mkdir(parents=True, exist_ok=True)
    paths = {run_dir / 'recipe.json'}
    if not reports_only and not folds:
        raise ValueError('Choose at least one completed fold.')
    if reports_only:
        selections = [p for p in run_dir.glob('selection_*') if (p / 'selection.json').is_file()]
        if not selections:
            raise ValueError('Finish development selection before exporting final reports.')
        for folder in selections:
            paths.update(p for p in folder.rglob('*') if p.is_file() and not p.name.endswith('.tmp'))
        name = run_dir.name + '_reports.zip'
    else:
        for fold in folds:
            stage = run_dir / 'projections/fixed_recipe' / f'fold{fold:02d}' / 'last.pt'
            saved = torch.load(stage, map_location='cpu', weights_only=True)
            if saved['epoch'] != saved['epochs_requested']:
                raise ValueError(f'Fold {fold} is not complete; retain its active checkpoint.')
            paths.add(run_dir / 'baselines' / f'fold{fold:02d}.pt')
            for folder in [run_dir / 'projections/fixed_recipe' / f'fold{fold:02d}',
                           run_dir / 'work' / f'fold{fold:02d}']:
                paths.update(p for p in folder.rglob('*') if p.is_file()
                             and not p.name.endswith(('.tmp', '_last.pt')))
        name = run_dir.name + '_' + '-'.join(f'fold{f:02d}' for f in folds) + '.zip'
    if not paths or any(not p.is_file() for p in paths):
        raise FileNotFoundError('A required completed-fold file is missing; no backup was written.')
    required = sum(p.stat().st_size for p in paths) + 100_000_000
    if shutil.disk_usage(export_dir).free < required:
        raise RuntimeError('Colab disk needs more space for the backup ZIP.')
    target = export_dir / name
    temporary = target.with_suffix('.zip.tmp')
    hashes = {str(Path(run_dir.name) / p.relative_to(run_dir)): sha256(p) for p in paths}
    with zipfile.ZipFile(temporary, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
        for p in sorted(paths):
            archive.write(p, str(Path(run_dir.name) / p.relative_to(run_dir)))
        archive.writestr('backup_checksums.json', json.dumps(hashes, indent=2))
    # Verify every archived byte against its source before advertising the backup.
    with zipfile.ZipFile(temporary) as archive:
        for name, expected in hashes.items():
            digest = hashlib.sha256()
            with archive.open(name) as stream:
                for block in iter(lambda: stream.read(1024 * 1024), b''):
                    digest.update(block)
            if digest.hexdigest() != expected:
                raise RuntimeError(f'ZIP verification failed: {name}')
    os.replace(temporary, target)
    print(f'Backup ready to download: {target} ({target.stat().st_size / 1e6:.1f} MB)')
    return target

### 2. Colab disk settings and import
`OUTPUT_ROOT`, `EXPORT_DIR`, and `LOG_PATH` must stay outside `/content/drive`. Disk capacity is measured on the Colab VM; no hard-coded Drive allowance is used. Imports are copies, not moves. Do not remove your Drive checkpoints until you have verified downloaded backups.

In [ ]:
FOLD_START = 1
FOLD_END = 25
BASELINE_EPOCHS = 15
PROJECTION_EPOCHS = 30
BATCH_SIZE = 8
SEED = 42
CONTRASTIVE_LOSS = 'supcon'
TEMPERATURE = 0.1
CHECKPOINT_EVERY = 1  # local Colab disk: keep one rolling last checkpoint
FOLDS_PER_RUN = 25  # Run All finishes remaining folds; set 1 to pause after each fold
RECIPE = 'cv04_focal_supcon_compact_v1'
DEVICE = 'cuda'

SAVED_DRIVE_RUN = DRIVE_DIR / 'runs' / RECIPE
OUTPUT_ROOT = Path('/content/ParkinsonRuns') / RECIPE
EXPORT_DIR = Path('/content/ParkinsonExports')
LOG_PATH = Path('/content/ParkinsonLogs') / f'{RECIPE}_training.log'
OUTPUT_ROOT.parent.mkdir(parents=True, exist_ok=True)
for path in (OUTPUT_ROOT, EXPORT_DIR, LOG_PATH):
    if not path.resolve().is_relative_to(Path('/content')) or path.resolve().is_relative_to(Path('/content/drive')):
        raise ValueError('This notebook writes only to Colab /content disk outside the Drive mount.')
if not 1 <= FOLD_START <= FOLD_END <= 25 or FOLDS_PER_RUN < 1:
    raise ValueError('Invalid fold range or folds-per-run.')
import_saved_run(SAVED_DRIVE_RUN, OUTPUT_ROOT)
# Also refresh paths after restoring downloaded ZIPs into this run directory.
if OUTPUT_ROOT.exists():
    rebase_backbone_paths(OUTPUT_ROOT)
    local_recipe = json.loads((OUTPUT_ROOT / 'recipe.json').read_text())
    local_recipe['baseline']['checkpoint_every'] = CHECKPOINT_EVERY
    (OUTPUT_ROOT / 'recipe.json').write_text(json.dumps(local_recipe, indent=2) + '\n')

from train_cv04_compact import storage_estimate
free_gb = shutil.disk_usage(OUTPUT_ROOT.parent).free / 1e9
budget = storage_estimate(OUTPUT_ROOT, list(range(FOLD_START, FOLD_END + 1)), free_gb=free_gb)
# Leave room for all fold ZIPs as well as the training files.
export_reserve = (FOLD_END - FOLD_START + 1) * 0.30
if free_gb < budget['additional_budget_gb'] + export_reserve:
    raise RuntimeError('Colab disk lacks the conservative allowance for training plus all export ZIPs.')
print(f'Colab disk free: {free_gb:.2f} GB; additional training/export allowance: '
      f"{budget['additional_budget_gb'] + export_reserve:.2f} GB")
print('Training device:', torch.cuda.get_device_name(0))
print('Training output:', OUTPUT_ROOT)
print('Downloads:', EXPORT_DIR)

### 3. Back up your already-completed folds
This prepares ZIPs for folds copied from your existing run. Download these first; a ZIP left on Colab disk is not a durable backup. Download through VS Code's mounted server Explorer. No Drive space is needed for these exports.

In [ ]:
finished = completed_folds(OUTPUT_ROOT, FOLD_START, FOLD_END, PROJECTION_EPOCHS)
for fold in finished:
    expected = EXPORT_DIR / f'{RECIPE}_fold{fold:02d}.zip'
    if not expected.is_file():
        export_backup(OUTPUT_ROOT, EXPORT_DIR, folds=[fold])
print('Completed folds:', finished)
print('Download the listed ZIPs from the Colab server to a folder on your Mac.')

### 4. Train remaining folds on Colab and create a backup after each fold
All computation stays on the Colab GPU. Progress bars show the full development schedule, stage epochs and batches. With the default `FOLDS_PER_RUN = 25`, the cell finishes all remaining folds and prepares each fold’s ZIP as it completes. Download those ZIPs while training continues. Set the value to 1 to pause for downloads. Rerunning skips fully completed folds. Completed folds are skipped. An unfinished readable checkpoint resumes; a damaged one uses the recovery checks below.

Only the active resumable backbone has optimizer/RNG state. At completion its exact full-precision weights are retained and the redundant optimizer checkpoint is removed **from the Colab run only**. Completed backbones remain available for CV07 reuse. Local saves every epoch can resume while this Colab VM still exists. After VM deletion, restore only the backups you actually downloaded; unfinished work not downloaded cannot be recovered.

Before training, the active checkpoint is loaded on CPU to verify that it is readable. If its archive is damaged, a readable matching Drive source is tried first. If no readable copy exists, the local unfinished work folder is preserved under `/content/ParkinsonQuarantine` and only that unfinished fold restarts. Completed backbones and Drive originals are not changed. Metadata mismatches and unrelated errors stop rather than being treated as corruption.


In [ ]:
from utils.cv04_notebook_progress import CV04ProgressState
from utils.notebook_training import run_with_progress

class ColabDiskProgress(CV04ProgressState):
    def update(self, event):
        super().update(event)
        if event['event'] == 'cv04_complete' and self.completed_folds < self.fold_end - self.fold_start + 1:
            self.phase = 'Batch finished — download its backup before continuing'

finished = completed_folds(OUTPUT_ROOT, FOLD_START, FOLD_END, PROJECTION_EPOCHS)
pending = [f for f in range(FOLD_START, FOLD_END + 1) if f not in finished]
if not pending:
    print('All requested folds are complete. Continue to KNN selection.')
for fold in pending[:FOLDS_PER_RUN]:
    prepare_active_checkpoint(OUTPUT_ROOT, SAVED_DRIVE_RUN, fold, MANIFEST,
                              Path('/content/ParkinsonQuarantine') / RECIPE)
    state = ColabDiskProgress('CV04 on Colab GPU', FOLD_START, FOLD_END, BASELINE_EPOCHS,
                              completed_folds=len(finished))
    command = [sys.executable, '-u', 'train_cv04_compact.py',
        '--manifest', str(MANIFEST), '--root', str(DATA_ROOT), '--output-dir', str(OUTPUT_ROOT),
        '--fold-start', str(fold), '--fold-end', str(fold),
        '--baseline-epochs', str(BASELINE_EPOCHS), '--projection-epochs', str(PROJECTION_EPOCHS),
        '--batch-size', str(BATCH_SIZE), '--checkpoint-every', str(CHECKPOINT_EVERY),
        '--seed', str(SEED), '--contrastive-loss', CONTRASTIVE_LOSS, '--temperature', str(TEMPERATURE),
        '--free-gb', str(shutil.disk_usage(OUTPUT_ROOT.parent).free / 1e9),
        '--device', 'cuda', '--progress-json']
    try:
        run_with_progress(command, title='CV04 on Colab GPU', fold_start=FOLD_START, fold_end=FOLD_END,
                          epochs=BASELINE_EPOCHS, progress_state=state, log_path=LOG_PATH)
    except subprocess.CalledProcessError as error:
        details = error.output or (LOG_PATH.read_text() if LOG_PATH.is_file() else str(error))
        raise RuntimeError('Underlying training error (full log: ' + str(LOG_PATH) + '):\n'
                           + brief_training_error(details)) from None
    export_backup(OUTPUT_ROOT, EXPORT_DIR, folds=[fold])
    finished.append(fold)
print(f'{len(finished)}/{FOLD_END - FOLD_START + 1} folds finished.')
print('Download new ZIPs from', EXPORT_DIR, 'before continuing.')

### 4. Select KNN settings on pooled development predictions
This compares k ∈ {1,3,5,7}, L2 distance, and uniform/distance weighting with the one trained contrastive recipe. It writes small selected projection/bank files and reports, referencing the **same** compact backbones. It never copies a backbone per KNN setting. Fold 22 has an explicit empty validation report.

Scores here are **development selection scores**, not unbiased test estimates. No locked-test evaluation or unspecified thesis weighted ensemble is run. A full result requires folds 1–25; a pilot is labeled incomplete. The shared training recipe is fixed across all folds.

This cell safely waits if any requested fold is absent or unfinished. No checkpoint hash is computed for a missing file. Rerun the training cell to continue, then rerun selection. With the default all-fold training setting, Run All reaches selection after training completes.


In [ ]:
K_VALUES = [1, 3, 5, 7]
DISTANCES = ['l2']
WEIGHTINGS = ['uniform', 'distance']
PROJECTIONS = OUTPUT_ROOT / 'projections'
# Name the selection by exact candidate hashes and grid, making reruns reusable.
source_files = [PROJECTIONS / 'fixed_recipe' / f'fold{f:02d}' / 'last.pt'
                for f in range(FOLD_START, FOLD_END + 1)]
SELECTION_DIR = None
selection = None
report = None
finished_for_selection = completed_folds(OUTPUT_ROOT, FOLD_START, FOLD_END, PROJECTION_EPOCHS)
missing_folds = [f for f in range(FOLD_START, FOLD_END + 1) if f not in finished_for_selection]
if missing_folds:
    print(f'Selection waiting: {len(finished_for_selection)}/{FOLD_END - FOLD_START + 1} folds complete.')
    print('Still needed:', missing_folds)
    print('Rerun the training cell to continue. Completed checkpoints are preserved.')
else:
    request = {'checkpoints': {str(p): sha256(p) for p in source_files},
               'folds': [FOLD_START, FOLD_END], 'k': K_VALUES,
               'distances': DISTANCES, 'weightings': WEIGHTINGS, 'manifest': sha256(MANIFEST)}
    signature = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()[:12]
    base_dir = OUTPUT_ROOT / f'selection_{signature}'
    SELECTION_DIR = base_dir
    attempt = 0
    # Preserve partial output after an interrupted selection and retry in a small fresh directory.
    while SELECTION_DIR.exists() and any(SELECTION_DIR.iterdir()) and not (SELECTION_DIR / 'selection.json').is_file():
        attempt += 1
        SELECTION_DIR = base_dir.with_name(base_dir.name + f'_attempt{attempt}')
    if not (SELECTION_DIR / 'selection.json').is_file():
        subprocess.run([sys.executable, '-u', 'tune_contrastive_knn.py',
            '--runs-dir', str(PROJECTIONS), '--manifest', str(MANIFEST), '--root', str(DATA_ROOT),
            '--output-dir', str(SELECTION_DIR), '--fold-start', str(FOLD_START), '--fold-end', str(FOLD_END),
            '--device', DEVICE, '--feature-batch-size', str(BATCH_SIZE),
            '--k-values', *map(str, K_VALUES), '--distances', *DISTANCES, '--weightings', *WEIGHTINGS], check=True)
    selection = json.loads((SELECTION_DIR / 'selection.json').read_text())
    report = json.loads((SELECTION_DIR / 'oof_cv04_report.json').read_text())
    print('Chosen KNN configuration:', selection['selected'])
    print('All 25 development folds:', selection['complete_development_schedule'])
    print('Expert image metrics:', report['image_expert_metrics'])
    print('Clinical participant metrics:', report['participant_clinical_metrics'])
    print('Reports:', SELECTION_DIR)

### 5. Verify a saved model and show storage used
Reload the first selected model, reproduce its validation predictions, and report actual run size. The weights, projection, KNN bank, class mapping, preprocessing and manifest checks all participate in independent reload. Keep the complete run directory: the small projection files refer to their source backbone files by path and hash.

In [ ]:
selection_dir = globals().get('SELECTION_DIR')
if selection_dir is None or not (selection_dir / 'selection.json').is_file():
    print('Reload check waiting: complete training and selection first.')
else:
    checkpoint = SELECTION_DIR / f'fold{FOLD_START:02d}' / 'selected.pt'
    reload_dir = SELECTION_DIR / f'reloaded_fold{FOLD_START:02d}'
    subprocess.run([sys.executable, '-u', 'eval_representation.py',
        '--checkpoint', str(checkpoint), '--manifest', str(MANIFEST), '--root', str(DATA_ROOT),
        '--device', DEVICE, '--output-dir', str(reload_dir)], check=True)
    assert (checkpoint.parent / 'val_predictions.csv').read_bytes() == (reload_dir / 'val_predictions.csv').read_bytes()
    print('Saved model reloaded successfully and reproduced validation predictions.')
    for folder in ['baselines', 'projections', 'work']:
        size = sum(p.stat().st_size for p in (OUTPUT_ROOT / folder).rglob('*') if p.is_file())
        print(f'{folder}: {size / 1e9:.3f} GB')
    total = sum(p.stat().st_size for p in OUTPUT_ROOT.rglob('*') if p.is_file())
    print(f'Whole run: {total / 1e9:.3f} GB')
    print('Selected checkpoint:', checkpoint)

### 7. Download final reports and selected projections
This ZIP adds the selected small projection checkpoints, KNN settings and reports. It references the same backbones already present in the completed-fold ZIPs. Keep all downloaded fold ZIPs plus this final reports ZIP, the notebook and the shared dataset/code archives.

In [ ]:
selection_dir = globals().get('SELECTION_DIR')
if selection_dir is None or not (selection_dir / 'selection.json').is_file():
    print('Final reports export waiting: completed-fold ZIPs are already available; finish training and selection first.')
else:
    export_backup(OUTPUT_ROOT, EXPORT_DIR, reports_only=True)
    print('Download the reports ZIP and every completed-fold ZIP to your Mac.')

### Recovery after a Colab VM reset
Reconnect to a Colab GPU kernel in VS Code and rerun setup/helpers. Upload the downloaded fold ZIPs (and reports ZIP if available) to `/content/ParkinsonRestore/` using VS Code's **Upload to Colab** action. Run the recovery cell below **before** the settings/import cell; the settings cell then refreshes source paths and skips importing older Drive data. A partial collection of fold ZIPs is supported. Never overwrite newer Colab work with older archives.

In [ ]:
# Optional recovery, ONLY on a fresh Colab VM before importing or training.
# restore_dir = Path('/content/ParkinsonRestore')
# target_parent = Path('/content/ParkinsonRuns')
# if (target_parent / 'cv04_focal_supcon_compact_v1').exists():
#     raise RuntimeError('A local run already exists; do not overwrite newer work.')
# archives = sorted(restore_dir.glob('*.zip'))
# if not archives:
#     raise FileNotFoundError('Upload your downloaded backup ZIPs first.')
# for source in archives:
#     with zipfile.ZipFile(source) as archive:
#         checksums = json.loads(archive.read('backup_checksums.json'))
#         for name, expected in checksums.items():
#             target = (target_parent / name).resolve()
#             if not target.is_relative_to(target_parent.resolve()):
#                 raise ValueError(f'Unsafe backup path: {name}')
#             content = archive.read(name)
#             if hashlib.sha256(content).hexdigest() != expected:
#                 raise ValueError(f'Backup checksum failed: {name}')
#             if target.exists() and target.read_bytes() != content:
#                 raise ValueError(f'Conflicting backups: {target}')
#             target.parent.mkdir(parents=True, exist_ok=True)
#             target.write_bytes(content)
# print('Backups restored. Run the Colab disk settings/import cell next.')